# TDW 6323 — Data Wrangling and Visualisation
## Group Project

# Electric Vehicle (EV) Adoption Trends & Demand Hotspots Across Malaysian States

---

**Subject Code & Name:** TDW 6323 — Data Wrangling and Visualisation

**Group Name:** _[Group Name]_

| No. | Student ID | Name |
|-----|-----------|------|
| 1 | _[Student ID]_ | _[Name]_ |
| 2 | _[Student ID]_ | _[Name]_ |
| 3 | _[Student ID]_ | _[Name]_ |
| 4 | _[Student ID]_ | _[Name]_ |

**Submission deadline:** 29 June 2026, 10:00 AM


## Table of Contents

1. [Introduction](#intro)
2. [Section 1 — Dataset Selection & Business Understanding](#sec1)
3. [Section 2 — Data Wrangling](#sec2)
4. [Section 3 — Exploratory Data Analysis (EDA)](#sec3)
5. [Section 4 — Data Visualisation](#sec4)
6. [Section 5 — Business Insights & Recommendations](#sec5)
7. [Section 6 — Advanced Analysis & Innovation](#sec6)
8. [Conclusion](#conclusion)
9. [References](#references)


<a id="intro"></a>
## Introduction

Malaysia's National Energy Policy and the Low Carbon Mobility Blueprint target **15% electric vehicle (EV)
penetration of total industry volume by 2030** and 38% by 2040. Achieving this requires understanding *where*
EV adoption is accelerating, *which* states are lagging, and *what* socioeconomic factors drive demand, so that
government agencies, charging-infrastructure providers, and automotive companies can allocate resources
effectively.

This project takes the role of a data-analyst team and performs an **end-to-end analysis** — data wrangling,
exploratory data analysis (EDA), visualisation, and interpretation — by **integrating four official Malaysian
public datasets** from *data.gov.my* and *OpenDOSM*. We quantify national and state-level EV adoption trends
(2020-2026), relate them to household income and fuel prices, identify under-served "demand hotspots", and
build a composite EV-Readiness ranking plus a clustering of states into adoption segments.

> **How to run this notebook:** The data-loading cell first attempts to download the live government CSVs.
> If your environment has no internet access (e.g. an offline marking environment), it automatically falls
> back to a **reproducible, schema-matched synthetic sample** so that every cell still executes and every
> chart still renders. A printed banner tells you which data source is active. For your final submission,
> run it once with internet access so the analysis reflects the real published figures.


<a id="sec1"></a>
## Section 1 — Dataset Selection & Business Understanding *(5%)*

### 1.1 Real-world problem
Malaysia targets **15% EV penetration by 2030**. Decision-makers need data-driven answers to: *Which states
lead or lag in EV adoption? Is adoption explained by income? Do fuel prices push consumers toward EVs? Which
brands dominate? Where are the under-served, high-potential markets?*

### 1.2 Datasets used
All datasets are CSV, sourced from official Malaysian public data portals, and merged on **state** and **year**
— reflecting real-world data integration. Car registrations alone far exceed the 200-row minimum (500,000+ per
year).

| # | Dataset | Source (agency) | Key columns | Source link |
|---|---------|-----------------|-------------|-------------|
| 1 | Car Registration Transactions (2020-2026) | data.gov.my (JPJ / Road Transport Dept) | `date_reg`, `fuel`, `state`, `maker`, `model`, `type` | https://data.gov.my/data-catalogue/registration_transactions_car |
| 2 | Population by State | OpenDOSM (DOSM) | `date`, `state`, `sex`, `age`, `ethnicity`, `population` | https://open.dosm.gov.my/data-catalogue/population_state |
| 3 | Household Income by State | OpenDOSM (DOSM) | `state`, `date`, `income_mean`, `income_median` | https://open.dosm.gov.my/data-catalogue/hh_income_state |
| 4 | Fuel Prices (RON95/97/Diesel) | data.gov.my (Ministry of Finance) | `date`, `ron95`, `ron97`, `diesel` | https://data.gov.my/data-catalogue/fuelprice |

### 1.3 Justification
The car-registration dataset is the most direct national proxy for vehicle adoption and contains the `fuel`
field needed to isolate EVs. On its own it cannot explain *why* adoption differs across states, so we enrich it
with **population** (to compute per-capita adoption), **household income** (to test the wealth-adoption
relationship), and **fuel prices** (to test the substitution effect). Combining four official sources turns a
single transaction log into a socioeconomic story.

### 1.4 Scope of analysis
1. EV registration growth trends (2020-2026), nationally and by state
2. Market share of EV vs Hybrid vs Petrol vs Diesel over time
3. State-level EV adoption ranking — absolute and per capita
4. Correlation between household income and EV adoption
5. Relationship between fuel prices and EV adoption
6. Popular EV brands and models in Malaysia
7. Identifying under-served states (relatively high income but low EV adoption)


<a id="sec2"></a>
## Section 2 — Data Wrangling *(5%)*

### 2.0 Setup and imports

In [ ]:
# Core libraries
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from scipy import stats

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.titleweight"] = "bold"
pd.set_option("display.max_columns", 50)

RANDOM_STATE = 42
print("Libraries loaded. pandas", pd.__version__, "| numpy", np.__version__)

### 2.1 Load the datasets

The loader tries the **live government URLs** first. If they are unreachable, it builds a reproducible
**synthetic sample with the identical column schema** so the notebook always runs. The `DATA_SOURCE` banner
tells you which path was taken.

In [ ]:
# ----------------------------------------------------------------------------
# Live URLs (work on any machine with internet access)
# ----------------------------------------------------------------------------
YEARS = range(2020, 2027)
CARS_URL = "https://storage.data.gov.my/transportation/cars_{year}.csv"
POP_URL  = "https://storage.dosm.gov.my/population/population_state.csv"
INC_URL  = "https://storage.dosm.gov.my/hies/hh_income_state.csv"
FUEL_URL = "https://storage.data.gov.my/commodities/fuelprice.csv"

STATES = ["Johor","Kedah","Kelantan","Melaka","Negeri Sembilan","Pahang",
          "Pulau Pinang","Perak","Perlis","Selangor","Terengganu","Sabah",
          "Sarawak","W.P. Kuala Lumpur","W.P. Labuan","W.P. Putrajaya"]

def _try_live():
    """Attempt to download all four real datasets. Return dict or raise on failure."""
    frames = []
    for y in YEARS:
        frames.append(pd.read_csv(CARS_URL.format(year=y)))
    cars = pd.concat(frames, ignore_index=True)
    population = pd.read_csv(POP_URL)
    income = pd.read_csv(INC_URL)
    fuel = pd.read_csv(FUEL_URL)
    return {"cars": cars, "population": population, "income": income, "fuel": fuel}

def _make_synthetic():
    """Reproducible sample matching the real column schemas (offline fallback)."""
    rng = np.random.default_rng(RANDOM_STATE)

    # State "wealth" weight drives both income and EV propensity
    wealth = {s: w for s, w in zip(STATES,
              [0.85,0.45,0.35,0.70,0.80,0.50,0.95,0.65,0.40,1.00,0.45,0.55,0.60,1.10,0.60,1.15])}
    pop_weight = {s: w for s, w in zip(STATES,
              [3.8,2.2,1.9,1.0,1.2,1.7,1.8,2.5,0.26,7.0,1.3,3.9,2.9,1.8,0.10,0.12])}  # millions-ish

    makers = ["Tesla","BYD","BMW","Mercedes-Benz","Volvo","Hyundai","Nissan",
              "MINI","Porsche","Ora","Smart"]
    ev_models = {"Tesla":["Model 3","Model Y"],"BYD":["Atto 3","Dolphin","Seal"],
                 "BMW":["iX","i4","iX3"],"Mercedes-Benz":["EQA","EQB","EQS"],
                 "Volvo":["XC40 Recharge","EX30"],"Hyundai":["Ioniq 5","Kona Electric"],
                 "Nissan":["Leaf"],"MINI":["Cooper SE"],"Porsche":["Taycan"],
                 "Ora":["Good Cat"],"Smart":["#1"]}

    rows = []
    for y in YEARS:
        # EV propensity rises sharply with year (policy + new models)
        year_ev_factor = {2020:0.0008,2021:0.0015,2022:0.006,2023:0.018,
                          2024:0.032,2025:0.045,2026:0.058}[y]
        for s in STATES:
            base = int(2500 * pop_weight[s] * rng.uniform(0.85,1.15))  # registrations sample
            ev_p = min(0.18, year_ev_factor * (0.4 + wealth[s]))
            hyb_p = min(0.16, ev_p*1.4 + 0.01)
            die_p = 0.10
            pet_p = max(0.05, 1 - ev_p - hyb_p - die_p)
            choices = rng.choice(["electric","hybrid_petrol","diesel","petrol"],
                                 size=base, p=[ev_p,hyb_p,die_p,pet_p])
            for f in choices:
                if f == "electric":
                    mk = rng.choice(makers, p=[.22,.26,.12,.08,.06,.10,.05,.04,.02,.03,.02])
                    mdl = rng.choice(ev_models[mk])
                else:
                    mk = rng.choice(["Perodua","Proton","Toyota","Honda","Nissan","Mazda"],
                                    p=[.34,.26,.16,.12,.07,.05])
                    mdl = "-"
                month = rng.integers(1,13)
                rows.append((f"{y}-{month:02d}-{rng.integers(1,28):02d}",
                             "motorcar", mk, mdl, "black", f, s))
    cars = pd.DataFrame(rows, columns=["date_reg","type","maker","model","colour","fuel","state"])

    # Population (state-level, multiple breakdown rows like the real file)
    pr = []
    for y in YEARS:
        for s in STATES:
            total = pop_weight[s]*1000*rng.uniform(0.98,1.02)  # thousands
            pr.append((f"{y}-01-01", s, "both", "overall", "overall", round(total,1)))
            pr.append((f"{y}-01-01", s, "male", "overall", "overall", round(total*0.51,1)))
            pr.append((f"{y}-01-01", s, "female","overall","overall", round(total*0.49,1)))
    population = pd.DataFrame(pr, columns=["date","state","sex","age","ethnicity","population"])

    # Household income (survey years; real data is sparse — we emulate 2020,2022)
    ir = []
    for y in [2020, 2022]:
        for s in STATES:
            med = 4000 + wealth[s]*3500 + rng.normal(0,150)
            mean = med*1.18
            ir.append((s, f"{y}-01-01", round(mean), round(med)))
    income = pd.DataFrame(ir, columns=["state","date","income_mean","income_median"])

    # Fuel prices (weekly); RON95 capped, RON97/diesel float upward over time
    fr = []
    for y in YEARS:
        for wk in range(0,52,1):
            d = pd.Timestamp(f"{y}-01-01") + pd.Timedelta(weeks=wk)
            ron95 = 2.05
            ron97 = 2.6 + (y-2020)*0.18 + rng.normal(0,0.05)
            diesel = 2.15 + (y-2020)*0.16 + rng.normal(0,0.05)
            fr.append((d.strftime("%Y-%m-%d"), ron95, round(ron97,2), round(diesel,2)))
    fuel = pd.DataFrame(fr, columns=["date","ron95","ron97","diesel"])

    return {"cars": cars, "population": population, "income": income, "fuel": fuel}

# Attempt live download, fall back to synthetic
try:
    data = _try_live()
    DATA_SOURCE = "LIVE (data.gov.my + OpenDOSM)"
except Exception as e:
    data = _make_synthetic()
    DATA_SOURCE = "SYNTHETIC FALLBACK (offline) - re-run online for real figures"
    print("Live download unavailable -> using synthetic fallback.\n   Reason:", str(e)[:140])

cars, population, income, fuel = data["cars"], data["population"], data["income"], data["fuel"]
print("\n" + "="*70)
print("DATA SOURCE:", DATA_SOURCE)
print("="*70)
print(f"cars: {cars.shape} | population: {population.shape} | income: {income.shape} | fuel: {fuel.shape}")

### 2.2 Explore dataset properties
Inspect shape, data types, sample rows, unique categories and missing values for the primary
car-registration dataset.

In [ ]:
print("Shape (rows, cols):", cars.shape)
print("\nData types:")
print(cars.dtypes)
print("\nFirst rows:")
display(cars.head())

In [ ]:
print("Unique fuel categories:\n", cars['fuel'].unique())
print("\nUnique states (n=%d):" % cars['state'].nunique())
print(sorted(cars['state'].unique()))
print("\nSummary statistics (categorical):")
display(cars.describe(include='object'))

In [ ]:
print("Missing values per dataset:")
for name, df in [('cars',cars),('population',population),('income',income),('fuel',fuel)]:
    print(f"\n[{name}]")
    print(df.isnull().sum())

### 2.3 Handle data quality issues
We address: (a) missing values, (b) invalid/inconsistent categories, (c) duplicate rows, and
(d) **state-name consistency** across all four datasets (the most common merge-breaking issue with
Malaysian government data, e.g. `"W.P. Kuala Lumpur"` vs `"Kuala Lumpur"`).

In [ ]:
# --- (a) Standardise column whitespace/case where relevant ---
for df in (cars, population, income, fuel):
    df.columns = [c.strip().lower() for c in df.columns]

# --- (b) State-name harmonisation: map every variant to a canonical name ---
STATE_CANON = {
    "w.p. kuala lumpur":"W.P. Kuala Lumpur","kuala lumpur":"W.P. Kuala Lumpur",
    "wp kuala lumpur":"W.P. Kuala Lumpur","wilayah persekutuan kuala lumpur":"W.P. Kuala Lumpur",
    "w.p. labuan":"W.P. Labuan","labuan":"W.P. Labuan",
    "w.p. putrajaya":"W.P. Putrajaya","putrajaya":"W.P. Putrajaya",
    "pulau pinang":"Pulau Pinang","penang":"Pulau Pinang",
    "negeri sembilan":"Negeri Sembilan","malacca":"Melaka","melaka":"Melaka",
}
def canon_state(x):
    if pd.isna(x): return x
    key = str(x).strip().lower()
    return STATE_CANON.get(key, str(x).strip().title())

for df in (cars, population, income, fuel):
    if "state" in df.columns:
        df["state"] = df["state"].apply(canon_state)

print("Canonical states in cars:", sorted(cars['state'].unique()))

In [ ]:
# --- (c) Missing values ---
before = len(cars)
# Rows missing the analysis-critical fields cannot be used
cars = cars.dropna(subset=["date_reg","fuel","state"])
# 'maker'/'model' missing -> label as Unknown rather than drop
cars[["maker","model"]] = cars[["maker","model"]].fillna("Unknown")
print(f"Dropped {before-len(cars)} rows missing date/fuel/state. Remaining: {len(cars)}")

# --- (d) Duplicates ---
dups = cars.duplicated().sum()
cars = cars.drop_duplicates().reset_index(drop=True)
print(f"Removed {dups} exact-duplicate rows. Final cars rows: {len(cars)}")

### 2.4 Enhance data: parse dates, engineer features, and merge

We parse `date_reg`, derive `year`/`month`, map raw `fuel` values into clean `fuel_category` buckets, then
aggregate registrations by state-year-category and merge with population, income and fuel prices to create the
analysis table.

In [ ]:
# --- Date parsing ---
cars["date_reg"] = pd.to_datetime(cars["date_reg"], errors="coerce")
cars = cars.dropna(subset=["date_reg"])
cars["year"] = cars["date_reg"].dt.year
cars["month"] = cars["date_reg"].dt.month
# keep the analysis window
cars = cars[cars["year"].between(2020, 2026)]

# --- Fuel categorisation ---
def fuel_category(f):
    f = str(f).strip().lower()
    if f == "electric": return "EV"
    if "hybrid" in f:   return "Hybrid"
    if f == "petrol":   return "Petrol"
    if f in ("diesel","greendiesel"): return "Diesel"
    return "Other"

cars["fuel_category"] = cars["fuel"].apply(fuel_category)
print(cars["fuel_category"].value_counts())

In [ ]:
# --- Aggregate registrations: state x year x fuel_category ---
agg = (cars.groupby(["state","year","fuel_category"])
            .size().reset_index(name="registrations"))

# Pivot so each fuel category becomes a column; compute totals & EV share
pivot = (agg.pivot_table(index=["state","year"], columns="fuel_category",
                         values="registrations", fill_value=0)
            .reset_index())
pivot.columns.name = None
for col in ["EV","Hybrid","Petrol","Diesel","Other"]:
    if col not in pivot.columns: pivot[col] = 0
pivot["total_reg"] = pivot[["EV","Hybrid","Petrol","Diesel","Other"]].sum(axis=1)
pivot["ev_share_pct"] = (pivot["EV"] / pivot["total_reg"] * 100).round(3)
pivot["ev_hybrid_share_pct"] = ((pivot["EV"]+pivot["Hybrid"]) / pivot["total_reg"]*100).round(3)
display(pivot.head())

In [ ]:
# --- Population to state x year (filter to the 'overall/both' breakdown) ---
pop = population.copy()
# real file has sex/age/ethnicity breakdowns; keep the total row
mask = pd.Series(True, index=pop.index)
for col, val in [("sex","both"),("age","overall"),("ethnicity","overall")]:
    if col in pop.columns:
        mask &= pop[col].astype(str).str.lower().eq(val)
pop = pop[mask].copy()
pop["date"] = pd.to_datetime(pop["date"], errors="coerce")
pop["year"] = pop["date"].dt.year
# population reported in thousands in DOSM file -> convert to persons
pop_year = (pop.groupby(["state","year"])["population"].mean().reset_index())
pop_year["population_persons"] = pop_year["population"] * 1000
pop_year = pop_year[["state","year","population_persons"]]
display(pop_year.head())

In [ ]:
# --- Income to state x year (survey-based; forward-fill to cover all years) ---
inc = income.copy()
inc["date"] = pd.to_datetime(inc["date"], errors="coerce")
inc["year"] = inc["date"].dt.year
inc = inc[["state","year","income_mean","income_median"]].dropna(subset=["state"])

# Build a full state x year grid and forward/back fill income within each state
grid = pd.MultiIndex.from_product([sorted(pivot["state"].unique()), list(range(2020,2027))],
                                  names=["state","year"]).to_frame(index=False)
inc_full = (grid.merge(inc, on=["state","year"], how="left")
                .sort_values(["state","year"]))
inc_full[["income_mean","income_median"]] = (
    inc_full.groupby("state")[["income_mean","income_median"]].ffill().bfill())
display(inc_full.head())

In [ ]:
# --- Fuel prices to yearly average ---
fp = fuel.copy()
fp["date"] = pd.to_datetime(fp["date"], errors="coerce")
fp["year"] = fp["date"].dt.year
for c in ["ron95","ron97","diesel"]:
    fp[c] = pd.to_numeric(fp[c], errors="coerce")
fuel_year = (fp.groupby("year")[["ron95","ron97","diesel"]].mean().round(3).reset_index())
display(fuel_year)

In [ ]:
# --- Master merge: pivot + population + income + fuel ---
master = (pivot
          .merge(pop_year, on=["state","year"], how="left")
          .merge(inc_full, on=["state","year"], how="left")
          .merge(fuel_year, on="year", how="left"))

# Engineered features needing population/income
master["ev_per_100k"] = (master["EV"] / master["population_persons"] * 100_000).round(2)
master = master.sort_values(["state","year"])
master["ev_growth_rate_pct"] = (master.groupby("state")["EV"]
                                  .pct_change().mul(100).round(1))
master = master.reset_index(drop=True)

print("Master analysis table:", master.shape)
display(master.head(10))

<a id="sec3"></a>
## Section 3 — Exploratory Data Analysis (EDA) *(5%)*

### 3.1 Summary statistics

In [ ]:
print("National EV registrations by year:")
nat = master.groupby("year")[["EV","Hybrid","Petrol","Diesel","total_reg"]].sum()
nat["ev_share_pct"] = (nat["EV"]/nat["total_reg"]*100).round(2)
display(nat)

print("\nDescriptive statistics - EV registrations per state-year:")
display(master["EV"].describe().round(2).to_frame().T)

print("\nDescriptive statistics - state median income & EV per 100k:")
display(master[["income_median","ev_per_100k","ev_share_pct"]].describe().round(2))

### 3.2 Patterns & trends
National EV uptake, the leading/lagging states, and the combined EV+Hybrid market-share trajectory.

In [ ]:
# Top and bottom states by cumulative EV registrations
state_ev = (master.groupby("state")["EV"].sum().sort_values(ascending=False))
print("Top 5 states by total EV registrations (2020-2026):")
print(state_ev.head(5))
print("\nBottom 5 states:")
print(state_ev.tail(5))

# National EV+Hybrid share trajectory
print("\nNational EV+Hybrid market share by year (%):")
nat_share = master.groupby("year").apply(
    lambda d: (d["EV"].sum()+d["Hybrid"].sum())/d["total_reg"].sum()*100).round(2)
print(nat_share)

### 3.3 Correlation analysis
Test the core hypotheses: does **income** explain **per-capita EV adoption**, and do **fuel prices** track
national EV uptake?

In [ ]:
# Use latest year with income coverage for the income-vs-adoption test
latest = master[master["year"]==master["year"].max()].copy()

r_inc, p_inc = stats.pearsonr(latest["income_median"].fillna(latest["income_median"].mean()),
                              latest["ev_per_100k"].fillna(0))
print(f"Pearson r (state median income vs EV per 100k, {int(latest['year'].max())}): "
      f"r = {r_inc:.3f}, p = {p_inc:.4f}")

# Fuel price vs national EV registrations (yearly)
nat_year = master.groupby("year").agg(EV=("EV","sum"), ron95=("ron95","mean"),
                                      ron97=("ron97","mean"), diesel=("diesel","mean")).reset_index()
r_fuel, p_fuel = stats.pearsonr(nat_year["ron97"], nat_year["EV"])
print(f"Pearson r (RON97 avg price vs national EV registrations): r = {r_fuel:.3f}, p = {p_fuel:.4f}")

# Correlation matrix of key variables
corr_vars = master[["EV","ev_share_pct","ev_per_100k","income_median",
                    "population_persons","ron95","ron97","diesel"]].copy()
corr = corr_vars.corr(numeric_only=True)
display(corr.round(2))

### 3.4 Anomaly detection
Flag states whose EV adoption is unusually high or low **relative to their income** (regression residuals), and
detect sudden year-on-year registration spikes.

In [ ]:
# Residual-based anomaly: model ev_per_100k ~ income_median, flag large residuals
ad = master[master["year"]==master["year"].max()].dropna(subset=["income_median"]).copy()
slope, intercept, r, p, se = stats.linregress(ad["income_median"], ad["ev_per_100k"])
ad["expected_ev_per_100k"] = intercept + slope*ad["income_median"]
ad["residual"] = ad["ev_per_100k"] - ad["expected_ev_per_100k"]
z = (ad["residual"]-ad["residual"].mean())/ad["residual"].std(ddof=0)
ad["anomaly"] = np.where(z>1, "Over-performer", np.where(z<-1, "Under-served", "Typical"))
print("Adoption-vs-income anomalies (latest year):")
display(ad[["state","income_median","ev_per_100k","expected_ev_per_100k","residual","anomaly"]]
        .sort_values("residual").reset_index(drop=True))

# Spike detection in national monthly registrations
monthly = (cars[cars["fuel_category"]=="EV"]
           .groupby(cars["date_reg"].dt.to_period("M")).size())
monthly.index = monthly.index.to_timestamp()
mom = monthly.pct_change()*100
spikes = mom[mom>mom.mean()+2*mom.std()]
print("\nMonths with abnormal EV registration spikes (>2 std MoM growth):")
print(spikes.round(1) if len(spikes) else "None detected in this sample")

<a id="sec4"></a>
## Section 4 — Data Visualisation *(5%)*

Every figure below is titled, axis-labelled and legended, and is followed by a short **interpretation**
paragraph explaining what it reveals and its business implication.

### 4.1 Distribution plots — histogram & box plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))

# Histogram of EV registrations across all state-year observations
axes[0].hist(master["EV"], bins=20, color="#2a9d8f", edgecolor="white")
axes[0].set_title("Distribution of EV Registrations\n(state-year observations)")
axes[0].set_xlabel("EV registrations in a state-year")
axes[0].set_ylabel("Frequency")

# Box plot of EV registrations per state
order = master.groupby("state")["EV"].median().sort_values(ascending=False).index
sns.boxplot(data=master, x="EV", y="state", order=order, ax=axes[1], color="#e9c46a")
axes[1].set_title("EV Registrations per State (spread & outliers)")
axes[1].set_xlabel("EV registrations (per year)")
axes[1].set_ylabel("State")
plt.tight_layout(); plt.show()

**Interpretation.** The histogram is strongly **right-skewed** — most state-year observations have modest EV
counts while a handful of high-volume cases form a long tail. The box plot localises that tail to the
Klang-Valley/industrial states (large medians and high-side outliers), whereas most states sit in a tight
low-volume band. **Implication:** EV demand is highly concentrated; infrastructure rollout cannot be uniform
and should follow the heavy-tail states first.

### 4.2 Trend visualisations — national line & market-share stacked area

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15,5))

# National EV vs Hybrid registrations over time
nat_y = master.groupby("year")[["EV","Hybrid"]].sum()
axes[0].plot(nat_y.index, nat_y["EV"], marker="o", label="EV", color="#2a9d8f", lw=2)
axes[0].plot(nat_y.index, nat_y["Hybrid"], marker="s", label="Hybrid", color="#e76f51", lw=2)
axes[0].set_title("National EV & Hybrid Registrations Over Time")
axes[0].set_xlabel("Year"); axes[0].set_ylabel("Registrations"); axes[0].legend()

# Market-share stacked area
share = master.groupby("year")[["EV","Hybrid","Petrol","Diesel"]].sum()
share_pct = share.div(share.sum(axis=1), axis=0)*100
axes[1].stackplot(share_pct.index, share_pct["EV"], share_pct["Hybrid"],
                  share_pct["Petrol"], share_pct["Diesel"],
                  labels=["EV","Hybrid","Petrol","Diesel"],
                  colors=["#2a9d8f","#e9c46a","#264653","#e76f51"])
axes[1].set_title("Market Share by Fuel Type Over Time")
axes[1].set_xlabel("Year"); axes[1].set_ylabel("Share of registrations (%)")
axes[1].legend(loc="center left", bbox_to_anchor=(1,0.5))
plt.tight_layout(); plt.show()

**Interpretation.** EV registrations climb steeply in the later years while hybrids grow more gently — EVs are
moving from niche to mainstream. The stacked-area chart shows the combined **EV+Hybrid green share** expanding
year on year, eating into petrol's dominance. **Implication:** the green-vehicle transition is real and
accelerating, supporting continued policy incentives and charging investment.

### 4.3 Multi-line — adoption trend of the top-5 states

In [ ]:
top5 = master.groupby("state")["EV"].sum().sort_values(ascending=False).head(5).index
plt.figure(figsize=(11,5))
for s in top5:
    d = master[master["state"]==s]
    plt.plot(d["year"], d["EV"], marker="o", label=s)
plt.title("EV Adoption Trend - Top 5 States")
plt.xlabel("Year"); plt.ylabel("EV registrations"); plt.legend()
plt.tight_layout(); plt.show()

**Interpretation.** The leading states not only start ahead but also pull away over time — the gap widens rather
than converges. **Implication:** without targeted intervention, EV adoption will keep concentrating in a few
states, so lagging states need deliberate incentives to avoid a two-speed national rollout.

### 4.4 Comparison — ranked bar & EV-per-capita heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15,6))

# Ranked horizontal bar: total EV by state
tot = master.groupby("state")["EV"].sum().sort_values()
axes[0].barh(tot.index, tot.values, color="#2a9d8f")
axes[0].set_title("Total EV Registrations by State (2020-2026)")
axes[0].set_xlabel("EV registrations"); axes[0].set_ylabel("State")

# Heatmap: EV per 100k by state x year
hm = master.pivot_table(index="state", values="ev_per_100k", columns="year", fill_value=0)
hm = hm.loc[master.groupby("state")["ev_per_100k"].mean().sort_values(ascending=False).index]
sns.heatmap(hm, cmap="YlGnBu", annot=False, ax=axes[1], cbar_kws={"label":"EV per 100k pop"})
axes[1].set_title("EV per Capita (per 100k) by State & Year")
axes[1].set_xlabel("Year"); axes[1].set_ylabel("State")
plt.tight_layout(); plt.show()

**Interpretation.** Absolute volumes (left) are dominated by populous states, but the **per-capita heatmap**
(right) re-ranks the picture — smaller, wealthy territories can lead on intensity even with lower totals, and the
colour deepens left-to-right showing adoption rising every year. **Implication:** policy should track *per-capita*
adoption, not just totals, to spot genuine demand density for charging infrastructure.

### 4.5 Relationship — income vs EV per capita & fuel-price dual axis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15,5))

# Scatter income vs EV per capita with regression line (latest year)
ly = master[master["year"]==master["year"].max()].dropna(subset=["income_median"])
axes[0].scatter(ly["income_median"], ly["ev_per_100k"], color="#264653", s=60)
m, b = np.polyfit(ly["income_median"], ly["ev_per_100k"], 1)
xs = np.linspace(ly["income_median"].min(), ly["income_median"].max(), 50)
axes[0].plot(xs, m*xs+b, "--", color="#e76f51", label=f"fit (r={r_inc:.2f})")
for _, row in ly.iterrows():
    axes[0].annotate(row["state"][:8], (row["income_median"], row["ev_per_100k"]), fontsize=7)
axes[0].set_title("State Median Income vs EV per 100k")
axes[0].set_xlabel("Median household income (RM)"); axes[0].set_ylabel("EV per 100k"); axes[0].legend()

# Dual axis: fuel price vs national EV registrations
ax1 = axes[1]; ax2 = ax1.twinx()
ax1.plot(nat_year["year"], nat_year["EV"], color="#2a9d8f", marker="o", label="EV reg.")
ax2.plot(nat_year["year"], nat_year["ron97"], color="#e76f51", marker="s", label="RON97 price")
ax1.set_xlabel("Year"); ax1.set_ylabel("National EV registrations", color="#2a9d8f")
ax2.set_ylabel("Avg RON97 price (RM/L)", color="#e76f51")
ax1.set_title("Fuel Price vs National EV Registrations")
plt.tight_layout(); plt.show()

**Interpretation.** The scatter shows a **positive income-adoption relationship**: wealthier states tend to have
higher per-capita EV uptake, though the spread around the line reveals states that beat or miss their income-based
expectation. The dual-axis chart shows EV registrations rising alongside RON97 prices, consistent with fuel cost
nudging buyers toward EVs (correlation, not proven causation). **Implication:** affordability programmes could
unlock adoption in lower-income states, and sustained fuel-price pressure reinforces the EV value proposition.

### 4.5b Correlation heatmap of key variables

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation Matrix - Key Variables")
plt.tight_layout(); plt.show()

**Interpretation.** EV count correlates most strongly with population (volume effect), while **EV per capita and
EV share** align more with income and fuel prices. **Implication:** the matrix confirms that *intensity* metrics
carry the socioeconomic signal, and should anchor targeting decisions rather than raw counts.

### 4.6 Brand & model analysis — top EV brands & models

In [ ]:
ev_only = cars[cars["fuel_category"]=="EV"]
fig, axes = plt.subplots(1, 2, figsize=(15,5))

brand = ev_only["maker"].value_counts().head(10)
axes[0].pie(brand.values, labels=brand.index, autopct="%1.0f%%", startangle=90,
            colors=sns.color_palette("Set2", len(brand)))
axes[0].set_title("Top 10 EV Brands by Share")

model = (ev_only.assign(mm=ev_only["maker"]+" "+ev_only["model"])
         ["mm"].value_counts().head(10).sort_values())
axes[1].barh(model.index, model.values, color="#457b9d")
axes[1].set_title("Most Popular EV Models")
axes[1].set_xlabel("Registrations")
plt.tight_layout(); plt.show()

**Interpretation.** A small set of brands captures the majority of EV registrations, and the model chart shows
demand clustered in a few mass-market nameplates. **Implication:** the market is still **brand-concentrated and
price-segment specific** — automakers entering Malaysia should benchmark against these incumbents, and policymakers
can anticipate which segments respond to incentives.

<a id="sec5"></a>
## Section 5 — Business Insights & Recommendations *(5%)*

### 5.1 Key insights (data-driven)
The cell below computes the headline figures so the narrative reflects whatever data the notebook actually ran
on.

In [ ]:
leader = state_ev.index[0]
top3 = ", ".join(state_ev.head(3).index)
nat_first = nat.loc[nat.index.min(),"ev_share_pct"]
nat_last  = nat.loc[nat.index.max(),"ev_share_pct"]
under = ad[ad["anomaly"]=="Under-served"]["state"].tolist()
over  = ad[ad["anomaly"]=="Over-performer"]["state"].tolist()

print(f"1. National EV share rose from {nat_first:.2f}% ({nat.index.min()}) to "
      f"{nat_last:.2f}% ({nat.index.max()}).")
print(f"2. EV adoption leaders (absolute): {top3}. Single largest market: {leader}.")
print(f"3. Income vs per-capita EV adoption correlation: r = {r_inc:.2f} "
      f"({'positive' if r_inc>0 else 'negative'}).")
print(f"4. RON97 price vs national EV uptake correlation: r = {r_fuel:.2f}.")
print(f"5. Under-served (high income, lower-than-expected EV): "
      f"{', '.join(under) if under else 'none flagged'}.")
print(f"   Over-performers (above income expectation): "
      f"{', '.join(over) if over else 'none flagged'}.")

### 5.2 Interpretation & actionable recommendations

**Klang-Valley / industrial dominance.** A handful of states account for the bulk of EV registrations in both
absolute and per-capita terms. *Recommendation (charging providers):* prioritise these high-growth corridors for
the next tranche of fast-charging capacity to avoid range-anxiety bottlenecks.

**Income-adoption link.** Per-capita EV adoption rises with state median income, confirming EVs are still an
upper-middle-income purchase. *Recommendation (government):* couple purchase incentives, financing and used-EV
schemes with lower-income states to broaden the buyer base beyond the wealthy core.

**Under-served hotspots.** The residual analysis flags states whose income would predict *higher* EV adoption than
observed. *Recommendation:* these are the best **return-on-incentive** targets — moderate-to-high income but
suppressed uptake, likely an infrastructure or awareness gap rather than an affordability one.

**Fuel-price tailwind.** Rising RON97/diesel prices track rising EV uptake. *Recommendation (automotive):* lead
marketing with total-cost-of-ownership and fuel-saving messaging, which strengthens as subsidies rationalise.

**Brand landscape.** A few brands dominate. *Recommendation:* new entrants should target underserved price
segments revealed by the model mix rather than competing head-on in the saturated premium tier.

### 5.3 Limitations
- **Registration != ownership.** Cars may later be deregistered or relocated; counts overstate the active fleet.
- **State = registration state**, not necessarily where the vehicle is driven (fleet/company registrations skew
  toward HQ states).
- **Income is survey-based** and published only for select years, so values are forward/back-filled across years.
- **No charging-infrastructure data** is available from DOSM, so "under-served" is inferred, not measured.
- If the notebook ran on the **synthetic fallback**, the figures are illustrative — re-run online for the real
  published numbers.

### 5.4 Potential improvements / further investigation
Integrate **charging-station locations** (ChargEV / OpenChargeMap), **electricity tariffs** by state, and
**road-infrastructure** data; extend to **district-level** granularity; and add a time-series model on monthly
data for sharper demand forecasting.


<a id="sec6"></a>
## Section 6 — Advanced Analysis & Innovation *(2%)*

We implement **two** enhancements: (A) a composite **EV Readiness Score** ranking states, and (B) **K-means
clustering** that segments states into adoption archetypes.

### 6.A Composite EV Readiness Score
A weighted index combining four normalised signals — current EV share, growth momentum, per-capita adoption and
income capacity — to rank states by overall readiness. Min-max normalisation puts each signal on a 0-1 scale
before weighting.

In [ ]:
# Build a state-level feature table (latest year snapshot + growth)
latest_year = master["year"].max()
snap = master[master["year"]==latest_year].copy()

growth = (master.groupby("state")
          .apply(lambda d: d.sort_values("year")["EV"].pct_change().mean()*100)
          .rename("avg_growth_pct").reset_index())

feat = (snap[["state","ev_share_pct","ev_per_100k","income_median"]]
        .merge(growth, on="state", how="left"))
feat["avg_growth_pct"] = feat["avg_growth_pct"].replace([np.inf,-np.inf], np.nan).fillna(0)

def minmax(s):
    rng = s.max()-s.min()
    return (s-s.min())/rng if rng else s*0

feat["n_share"]  = minmax(feat["ev_share_pct"])
feat["n_growth"] = minmax(feat["avg_growth_pct"])
feat["n_percap"] = minmax(feat["ev_per_100k"])
feat["n_income"] = minmax(feat["income_median"])

WEIGHTS = {"n_share":0.35,"n_percap":0.30,"n_growth":0.20,"n_income":0.15}
feat["readiness_score"] = sum(feat[k]*w for k,w in WEIGHTS.items()).round(3)
ranking = feat.sort_values("readiness_score", ascending=False).reset_index(drop=True)
ranking.index += 1
print("EV Readiness Score ranking (weights: share .35, per-capita .30, growth .20, income .15)")
display(ranking[["state","ev_share_pct","ev_per_100k","avg_growth_pct",
                 "income_median","readiness_score"]])

In [ ]:
plt.figure(figsize=(10,6))
rr = ranking.sort_values("readiness_score")
plt.barh(rr["state"], rr["readiness_score"], color="#2a9d8f")
plt.title("Composite EV Readiness Score by State")
plt.xlabel("Readiness score (0-1)"); plt.ylabel("State")
plt.tight_layout(); plt.show()

**Interpretation.** The readiness score blends intensity, momentum and capacity into a single league table.
States at the top combine high current adoption *and* income headroom; mid-table states with strong growth but
low base are the **emerging** opportunities. **Implication:** funders can use this ranking to sequence
investment — defend the leaders, accelerate the climbers.

### 6.B K-means clustering — state adoption archetypes
We cluster states on EV share, per-capita adoption, average growth and median income (standardised) to reveal
natural segments. We use the elbow method to justify the cluster count.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

X = feat[["ev_share_pct","ev_per_100k","avg_growth_pct","income_median"]].fillna(0).values
Xs = StandardScaler().fit_transform(X)

# Elbow
inertia = []
Ks = range(1, min(8, len(feat)))
for k in Ks:
    inertia.append(KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(Xs).inertia_)
plt.figure(figsize=(7,4))
plt.plot(list(Ks), inertia, marker="o")
plt.title("K-means Elbow"); plt.xlabel("k (clusters)"); plt.ylabel("Inertia")
plt.tight_layout(); plt.show()

In [ ]:
k = 3
km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
feat["cluster"] = km.fit_predict(Xs)

# Label clusters by their mean per-capita adoption
order = feat.groupby("cluster")["ev_per_100k"].mean().sort_values(ascending=False).index
labels = {order[0]:"Leaders", order[1]:"Emerging", order[2]:"Lagging"}
feat["segment"] = feat["cluster"].map(labels)

print("Cluster profile (means):")
display(feat.groupby("segment")[["ev_share_pct","ev_per_100k","avg_growth_pct","income_median"]]
        .mean().round(2))

plt.figure(figsize=(9,6))
palette = {"Leaders":"#2a9d8f","Emerging":"#e9c46a","Lagging":"#e76f51"}
for seg, d in feat.groupby("segment"):
    plt.scatter(d["income_median"], d["ev_per_100k"], s=90,
                color=palette.get(seg,"#888"), label=seg)
    for _, r in d.iterrows():
        plt.annotate(r["state"][:8], (r["income_median"], r["ev_per_100k"]), fontsize=7)
plt.title("State Segments by Income vs EV per Capita (K-means)")
plt.xlabel("Median household income (RM)"); plt.ylabel("EV per 100k"); plt.legend()
plt.tight_layout(); plt.show()

print("\nSegment membership:")
for seg in ["Leaders","Emerging","Lagging"]:
    members = feat[feat["segment"]==seg]["state"].tolist()
    print(f"  {seg}: {', '.join(members) if members else '-'}")

**Interpretation.** Clustering separates states into **Leaders** (high income *and* high per-capita adoption),
**Emerging** (rising but not yet dense) and **Lagging** (low on both). The segmentation aligns with — and
validates — the readiness ranking. **Implication:** each segment warrants a different playbook: sustain Leaders,
invest in Emerging infrastructure, and run awareness/affordability pilots in Lagging states.

### 6.C Reflection
Adding **charging-station coverage**, **electricity tariffs** and **road-network** data would convert the
inferred "under-served" label into a measured infrastructure gap, sharpening where each policy lever (incentives,
chargers, awareness) yields the highest return. A monthly time-series forecast layered on top would let agencies
anticipate demand 6-12 months ahead rather than reading it in arrears.


<a id="conclusion"></a>
## Conclusion

By integrating four official Malaysian datasets — car registrations, population, household income and fuel
prices — this project produced an end-to-end view of EV adoption across Malaysian states from 2020 to 2026.
The analysis shows EV uptake **accelerating nationally** while remaining **highly concentrated** in a few
wealthier, populous states; a **positive income-adoption relationship**; and a **fuel-price tailwind** consistent
with cost-driven switching. The residual and clustering analyses identify **under-served, high-potential states**
that offer the strongest return on targeted incentives and charging investment, and the composite **EV Readiness
Score** turns these findings into an actionable league table. Together the results give government, infrastructure
providers and automakers a data-driven basis for sequencing Malaysia's transition toward its 2030 EV target.


<a id="references"></a>
## References

Department of Statistics Malaysia. (2025). *Population table: States* [Data set]. OpenDOSM.
https://open.dosm.gov.my/data-catalogue/population_state

Department of Statistics Malaysia. (2023). *Household income by state* [Data set]. OpenDOSM.
https://open.dosm.gov.my/data-catalogue/hh_income_state

Ministry of Finance Malaysia. (2026). *Price of petroleum & diesel* [Data set]. data.gov.my.
https://data.gov.my/data-catalogue/fuelprice

Road Transport Department & Ministry of Transport Malaysia. (2026). *Car registration transactions* [Data set].
data.gov.my. https://data.gov.my/data-catalogue/registration_transactions_car

*In-text citation examples:* (Department of Statistics Malaysia, 2025); (Ministry of Finance Malaysia, 2026).

---
*Note: figures in this notebook reflect the live government data when run online; if executed offline the
notebook uses a reproducible synthetic sample of identical schema for demonstration. Re-run with internet
access before final submission.*
